# Preparación de datos

Se va a mostrar cómo preparar datos usando **Pandas**. Para esto se va a trabajar con un dataset que contiene datos de clientes de una empresa de telecomunicaciones.

El equipo de análisis cuenta con un dataset cuyo diccionario de datos se puede consultar en [IBM](https://community.ibm.com/community/user/businessanalytics/blogs/steven-macko/2019/07/11/telco-customer-churn-1113).

Se va a asumir que se va a usar este dataset para entrenar un modelo de ML que prediga la variable **Churn Value**.

In [2]:
from pathlib import Path

DATA_DIR = Path().resolve().parent / "data" / "raw"
data_file = "Telco_customer_churn.csv"
data_path = DATA_DIR / data_file

In [3]:
import pandas as pd

df = pd.read_csv(
    data_path,
    na_values=[' ']
    )
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


# Pre-selección de variables

Algunas tareas que se hacen en esta fase son:

- Elegir las variables y registros más relevantes.
- Excluir columnas irrelevantes, redundantes o de mala calidad.
- Filtrar por condiciones específicas.

Vamos a hacer una breve comprensión de los datos antes de abordar estas tareas.


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [5]:
df.describe()

,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,Churn Value,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7032.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,2283.300441,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,2266.771362,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,18.800000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,401.450000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,1397.475000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,3794.737500,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,8684.800000,1.000000,100.000000,6500.000000


In [6]:
df.describe(include='object').T

,count,unique,top,freq
CustomerID,7043,7043,3668-QPYBK,1
Country,7043,1,United States,7043
State,7043,1,California,7043
City,7043,1129,Los Angeles,305
Lat Long,7043,1652,"34.159534, -116.425984",5
Gender,7043,2,Male,3555
Senior Citizen,7043,2,No,5901
Partner,7043,2,No,3641
Dependents,7043,2,No,5416
Phone Service,7043,2,Yes,6361


In [7]:
df.isnull().sum().sort_values(ascending=False).head(5)

Churn Reason     5174
Total Charges      11
CustomerID          0
Count               0
Country             0
dtype: int64

In [8]:
df[['Total Charges', 'Churn Reason']].isnull().sum()*100/df.shape[0]

Total Charges     0.156183
Churn Reason     73.463013
dtype: float64

In [9]:
print('Número de registros duplicados: ', df.duplicated().sum())

Número de registros duplicados:  0


Hallazgos:

- **CustomerID** es una clave primaria.
- **Count**, **Country** y **State** son variables de valor único.
- En el 73.46% de las filas no hay datos válidos de la variable **Churn Reason**.
- La información que tiene la variable **Lat Long** es la misma que contienen las variables **Latitude** y **Longitude**.

Soluciones:
- Convertir **CustomerID** en índice, o eliminar la variable.
- Eliminar las variables **Count**, **Country**, **State** y **Lat Long**.

In [10]:
df = df.set_index('CustomerID')
df = df.drop(columns=['Count', 'Country', 'State', 'Lat Long'])
df.head()

,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
CustomerID,,,,,,,,,,,,,,,,,,,,,
3668-QPYBK,Los Angeles,90003,33.964131,-118.272783,Male,No,No,No,2,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
9237-HQITU,Los Angeles,90005,34.059281,-118.307420,Female,No,No,Yes,2,Yes,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
9305-CDSKC,Los Angeles,90006,34.048013,-118.293953,Female,No,No,Yes,8,Yes,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
7892-POOKP,Los Angeles,90010,34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
0280-XJGEX,Los Angeles,90015,34.039224,-118.266293,Male,No,No,Yes,49,Yes,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


# Limpieza de datos

Algunas tareas frecuentes de limpieza de datos son:

- Tratar valores faltantes.
- Corregir errores, inconsistencias o vacíos.
- Eliminar duplicados.

Los duplicados se eliminan con el método `drop_duplicates`:

In [11]:
df =  df.drop_duplicates()

## Gestión de datos nulos

Otro problema frecuente es el de los *datos nulos*.

Frente a esto, hay varias alternativas de solución:

1. Eliminar los variables que tienen datos nulos.
2. Eliminar los registros que tienen datos nulos.
3. Imputar datos, esto es, reemplazar el dato nulo por algún valor válido.

La elección de la alternativa de solución, o incluso la decisión de **no hacer nada** depende, como siempre, de los objetivos del análisis de datos, y en particular, del tipo de análisis de datos que se va a hacer.


### Eliminar variables con datos nulos

Esto se hace en 2 casos:

- La variable no es relevante.
- La variable tiene un porcentaje muy elevado de datos nulos (no hay un umbral fijo).

En nuestro caso, ya identificamos que la variable **Churn Reason** tiene un porcentaje muy alto de datos nulos, superior al 73%. Además, no tendría sentido utilizar esta variable para predecir **Churn Value**, porque la primera depende directamente de la segunda. Usar esta variable crearía un problema de [fuga de datos](https://en.wikipedia.org/wiki/Leakage_(machine_learning)). Por estas razones se procederá a eliminar esta variable:

In [12]:
df = df.drop(columns=['Churn Reason'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 3668-QPYBK to 3186-AJIEK
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   City               7043 non-null   object 
 1   Zip Code           7043 non-null   int64  
 2   Latitude           7043 non-null   float64
 3   Longitude          7043 non-null   float64
 4   Gender             7043 non-null   object 
 5   Senior Citizen     7043 non-null   object 
 6   Partner            7043 non-null   object 
 7   Dependents         7043 non-null   object 
 8   Tenure Months      7043 non-null   int64  
 9   Phone Service      7043 non-null   object 
 10  Multiple Lines     7043 non-null   object 
 11  Internet Service   7043 non-null   object 
 12  Online Security    7043 non-null   object 
 13  Online Backup      7043 non-null   object 
 14  Device Protection  7043 non-null   object 
 15  Tech Support       7043 non-null   object 
 16  Streaming TV  

### Eliminar registros con datos nulos

Los registros con datos nulos se pueden eliminar con el método `dropna()`. También podrían utilizarse otros métodos como `notnull()` o `notna()`.

Por defecto, `dropna` elimina los datos nulos de **todas** las variables:

In [13]:
df.dropna().info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 3668-QPYBK to 3186-AJIEK
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   City               7032 non-null   object 
 1   Zip Code           7032 non-null   int64  
 2   Latitude           7032 non-null   float64
 3   Longitude          7032 non-null   float64
 4   Gender             7032 non-null   object 
 5   Senior Citizen     7032 non-null   object 
 6   Partner            7032 non-null   object 
 7   Dependents         7032 non-null   object 
 8   Tenure Months      7032 non-null   int64  
 9   Phone Service      7032 non-null   object 
 10  Multiple Lines     7032 non-null   object 
 11  Internet Service   7032 non-null   object 
 12  Online Security    7032 non-null   object 
 13  Online Backup      7032 non-null   object 
 14  Device Protection  7032 non-null   object 
 15  Tech Support       7032 non-null   object 
 16  Streaming TV  

Sin embargo, también es posible seleccionar las variables a las cuales se les va a eliminar los datos nulos:

In [14]:
df = df.dropna(subset=['Total Charges'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 3668-QPYBK to 3186-AJIEK
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   City               7032 non-null   object 
 1   Zip Code           7032 non-null   int64  
 2   Latitude           7032 non-null   float64
 3   Longitude          7032 non-null   float64
 4   Gender             7032 non-null   object 
 5   Senior Citizen     7032 non-null   object 
 6   Partner            7032 non-null   object 
 7   Dependents         7032 non-null   object 
 8   Tenure Months      7032 non-null   int64  
 9   Phone Service      7032 non-null   object 
 10  Multiple Lines     7032 non-null   object 
 11  Internet Service   7032 non-null   object 
 12  Online Security    7032 non-null   object 
 13  Online Backup      7032 non-null   object 
 14  Device Protection  7032 non-null   object 
 15  Tech Support       7032 non-null   object 
 16  Streaming TV  

### Imputación de datos nulos

Imputar es asignar o rellenar el dato nulo con un dato válido.

En Pandas existen 3 funciones para hacer imputación de datos: `ffill()`, `bfill()`, y `fillna()`. Vamos a ver como funcionan estos métodos con un nuevo dataset:

In [15]:
data_file = "auto-mpg.data"
data_path = DATA_DIR / data_file

df = pd.read_csv(
    data_path,
    sep=r'\s+',
    header=None,
    names=['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin', 'car_name']
    )
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8.0,307.0,130.0,3504.0,12.0,70.0,1.0,chevrolet chevelle malibu
1,15.0,8.0,350.0,165.0,3693.0,11.5,70.0,1.0,buick skylark 320
2,18.0,8.0,318.0,150.0,3436.0,11.0,70.0,1.0,plymouth satellite
3,16.0,8.0,304.0,150.0,3433.0,12.0,70.0,1.0,amc rebel sst
4,17.0,8.0,302.0,140.0,3449.0,10.5,70.0,1.0,ford torino


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406 entries, 0 to 405
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     406 non-null    float64
 2   displacement  406 non-null    float64
 3   horsepower    400 non-null    float64
 4   weight        406 non-null    float64
 5   acceleration  406 non-null    float64
 6   model_year    406 non-null    float64
 7   origin        406 non-null    float64
 8   car_name      406 non-null    object 
dtypes: float64(8), object(1)
memory usage: 28.7+ KB


Este dataset tiene datos nulos en las variables **mpg** y **horsepower**. Los registros con datos nulos de **horsepower** tienen los siguientes índices:

In [17]:
hp_nulls = list(df[df['horsepower'].isnull()].index)
hp_nulls

[38, 133, 337, 343, 361, 382]

In [18]:
df.loc[38]

mpg                   25.0
cylinders              4.0
displacement          98.0
horsepower             NaN
weight              2046.0
acceleration          19.0
model_year            71.0
origin                 1.0
car_name        ford pinto
Name: 38, dtype: object

In [19]:
df.loc[37:39]

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
37,25.0,4.0,113.0,95.0,2228.0,14.0,71.0,3.0,toyota corona
38,25.0,4.0,98.0,NaN,2046.0,19.0,71.0,1.0,ford pinto
39,NaN,4.0,97.0,48.0,1978.0,20.0,71.0,2.0,volkswagen super beetle 117


`ffill()` imputa al dato nulo el mismo valor del registro *anterior*:

In [20]:
df['horsepower'].ffill().loc[37:39]

37    95.0
38    95.0
39    48.0
Name: horsepower, dtype: float64

Mientras, `bfill()`imputa al dato nulo el mismo valor del registro *siguiente*:

In [21]:
df['horsepower'].bfill().loc[37:39]

37    95.0
38    48.0
39    48.0
Name: horsepower, dtype: float64

Con `fillna()`es posible asignar un valor específico a los datos nulos:

In [22]:
df['horsepower'].fillna(value=100).loc[37:39]

37     95.0
38    100.0
39     48.0
Name: horsepower, dtype: float64

También es posible asignar un valor calculado, por ejemplo, el valor promedio de la variable a la que se le están imputando los datos nulos:

In [23]:
df.horsepower.mean()

np.float64(105.0825)

In [24]:
df['horsepower'].fillna(value=df['horsepower'].mean()).loc[37:39]

37     95.0000
38    105.0825
39     48.0000
Name: horsepower, dtype: float64

Por último, también es posible hacer imputación a valores calculados diferenciados por grupo. Por ejemplo, se va a asignar a cada dato nulo de **horsepower** la media de la variable, pero **dependiendo del valor de la variable origin**:

In [25]:
df.groupby('origin')['horsepower'].mean()

origin
1.0    119.900000
2.0     81.000000
3.0     79.835443
Name: horsepower, dtype: float64

In [26]:
df.loc[38]

mpg                   25.0
cylinders              4.0
displacement          98.0
horsepower             NaN
weight              2046.0
acceleration          19.0
model_year            71.0
origin                 1.0
car_name        ford pinto
Name: 38, dtype: object

In [27]:
df['horsepower'].fillna(df.groupby('origin')['horsepower'].transform(lambda x: x.mean())).loc[38]

np.float64(119.9)

In [28]:
df.loc[337]

mpg                             40.9
cylinders                        4.0
displacement                    85.0
horsepower                       NaN
weight                        1835.0
acceleration                    17.3
model_year                      80.0
origin                           2.0
car_name        renault lecar deluxe
Name: 337, dtype: object

In [29]:
df['horsepower'].fillna(df.groupby('origin')['horsepower'].transform(lambda x: x.mean())).loc[337]

np.float64(81.0)

### Corrección de errores en cadenas de texto

En Pandas es posible usar los métodos de [cadenas de texto](https://docs.python.org/es/3/library/stdtypes.html#string-methods) nativos de Python.

Por ejemplo, si se desea poner en mayúscula sostenida todos los nombres de los vehículos se puede usar el método `str.upper`:

In [30]:
df['car_name'] = df['car_name'].str.upper()
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8.0,307.0,130.0,3504.0,12.0,70.0,1.0,CHEVROLET CHEVELLE MALIBU
1,15.0,8.0,350.0,165.0,3693.0,11.5,70.0,1.0,BUICK SKYLARK 320
2,18.0,8.0,318.0,150.0,3436.0,11.0,70.0,1.0,PLYMOUTH SATELLITE
3,16.0,8.0,304.0,150.0,3433.0,12.0,70.0,1.0,AMC REBEL SST
4,17.0,8.0,302.0,140.0,3449.0,10.5,70.0,1.0,FORD TORINO


Otros métodos similares son `str.capitalize`, `str.lower`, y `str.title`:

In [31]:
df['car_name'] = df['car_name'].str.title()
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8.0,307.0,130.0,3504.0,12.0,70.0,1.0,Chevrolet Chevelle Malibu
1,15.0,8.0,350.0,165.0,3693.0,11.5,70.0,1.0,Buick Skylark 320
2,18.0,8.0,318.0,150.0,3436.0,11.0,70.0,1.0,Plymouth Satellite
3,16.0,8.0,304.0,150.0,3433.0,12.0,70.0,1.0,Amc Rebel Sst
4,17.0,8.0,302.0,140.0,3449.0,10.5,70.0,1.0,Ford Torino


El método `str.replace` permite reemplazar partes de las cadenas de texto. POr ejemplo, si se quisiera reemplazar los espacios en blanco por guiones se puede hacer esto:

In [32]:
df['car_name'] = df['car_name'].str.replace(' ', '-')
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8.0,307.0,130.0,3504.0,12.0,70.0,1.0,Chevrolet-Chevelle-Malibu
1,15.0,8.0,350.0,165.0,3693.0,11.5,70.0,1.0,Buick-Skylark-320
2,18.0,8.0,318.0,150.0,3436.0,11.0,70.0,1.0,Plymouth-Satellite
3,16.0,8.0,304.0,150.0,3433.0,12.0,70.0,1.0,Amc-Rebel-Sst
4,17.0,8.0,302.0,140.0,3449.0,10.5,70.0,1.0,Ford-Torino


También es posible extraer una parte de las cadenas de texto. Por ejemplo, se va a crear una nueva variable que sea la **marca** del carro, que es la primera palabra de **car_name**:

In [33]:
df['car_brand'] = df['car_name'].str.split('-').str[0]
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name,car_brand
0,18.0,8.0,307.0,130.0,3504.0,12.0,70.0,1.0,Chevrolet-Chevelle-Malibu,Chevrolet
1,15.0,8.0,350.0,165.0,3693.0,11.5,70.0,1.0,Buick-Skylark-320,Buick
2,18.0,8.0,318.0,150.0,3436.0,11.0,70.0,1.0,Plymouth-Satellite,Plymouth
3,16.0,8.0,304.0,150.0,3433.0,12.0,70.0,1.0,Amc-Rebel-Sst,Amc
4,17.0,8.0,302.0,140.0,3449.0,10.5,70.0,1.0,Ford-Torino,Ford


Una revisión exhaustiva de las marcas muestra que hay marcas escritas de forma incorrecta, o de forma diferente. Por ejemplo los vehículos de marca *Volkswagen* están etiquetados de 3 formas diferentes: "volkswagen", "vw", y "vokswagen". Este tipo de errores se puede corregir usando la función `replace`, que se basa en el método nativo de Python (`str.replace`), pero es más potente:

In [34]:
df['car_brand'].value_counts()

car_brand
Ford          53
Chevrolet     44
Plymouth      32
Amc           29
Dodge         28
Toyota        25
Datsun        23
Buick         17
Pontiac       16
Volkswagen    16
Honda         13
Mercury       11
Oldsmobile    10
Mazda         10
Peugeot        8
Fiat           8
Audi           7
Chrysler       6
Volvo          6
Vw             6
Saab           5
Renault        5
Opel           4
Subaru         4
Chevy          3
Mercedes       3
Cadillac       2
Bmw            2
Maxda          2
Hi             1
Citroen        1
Toyouta        1
Capri          1
Chevroelt      1
Vokswagen      1
Triumph        1
Nissan         1
Name: count, dtype: int64

In [35]:
df['car_brand'] = df['car_brand'].replace({'Vw': 'Volkswagen', 'Vokswagen': 'Volkswagen'})
df['car_brand'].value_counts()

car_brand
Ford          53
Chevrolet     44
Plymouth      32
Amc           29
Dodge         28
Toyota        25
Datsun        23
Volkswagen    23
Buick         17
Pontiac       16
Honda         13
Mercury       11
Oldsmobile    10
Mazda         10
Peugeot        8
Fiat           8
Audi           7
Chrysler       6
Volvo          6
Renault        5
Saab           5
Opel           4
Subaru         4
Mercedes       3
Chevy          3
Bmw            2
Maxda          2
Cadillac       2
Citroen        1
Hi             1
Toyouta        1
Capri          1
Chevroelt      1
Triumph        1
Nissan         1
Name: count, dtype: int64